In [9]:
import pyarrow.parquet as pq
import pandas as pd
import datetime



In [22]:
print(df["tpep_pickup_datetime"].min(),df["tpep_pickup_datetime"].max())

2008-12-31 23:03:20 2026-04-01 00:06:25


In [ ]:
df.filter((df["tpep_pickup_datetime"] >= "2026-03-01 00:00:00") & (df["tpep_pickup_datetime"] <= "2026-03-31 23:59:59"))

""
0
1
2
3
4
...
3952446
3952447
3952448
3952449


In [20]:
neg = df.filter(df["total_amount"]>0)

In [21]:
neg.count()

Series([], dtype: int64)

In [28]:
df.sort_values(df["total_amount"])

KeyError: 0          24.94
1          15.00
2          17.22
3          21.42
4          20.58
           ...  
3952446    60.37
3952447    13.82
3952448    52.70
3952449    53.93
3952450    12.94
Name: total_amount, Length: 3952451, dtype: float64

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.appName("Project_weather_NYC_taxi").master("local[*]").config("spark.sql.adaptive.enabled",True).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/14 17:31:53 WARN Utils: Your hostname, MacBook-Air-Andrij.local, resolves to a loopback address: 127.0.0.1; using 10.239.1.69 instead (on interface en0)
26/10/14 17:31:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Volumes/KINGSTON_ANDRII/DE_projects/end_to_end_NYC_taxi_weather_pipeline/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 17:31:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where a

In [6]:
df.filter(col("total_amount") < 0).count()

21305

In [14]:
df.filter(col("passenger_count") <= 0.0).count()

12967

In [11]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [20]:
import requests

In [4]:
import requests
res = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 40.7128, "longitude": -74.0060,
    "start_date": "2026-03-01", "end_date": "2026-03-31",
    "hourly": "temperature_2m,precipitation,weather_code",
    "timezone": "America/New_York",
}, timeout=30)

In [22]:
res.status_code

200

In [23]:
import sys
print(sys.executable)

/Volumes/KINGSTON_ANDRII/DE_projects/end_to_end_NYC_taxi_weather_pipeline/.venv/bin/python


In [5]:
data = res.json()

In [6]:
print(data["hourly"].keys())

dict_keys(['time', 'temperature_2m', 'precipitation', 'weather_code'])


PySparkTypeError: [CANNOT_INFER_SCHEMA_FOR_TYPE] Can not infer schema for type: `str`.

In [11]:
weather_df = pd.DataFrame(data["hourly"])
print(weather_df)


                 time  temperature_2m  precipitation  weather_code
0    2026-03-01T00:00            -2.6            0.0             3
1    2026-03-01T01:00            -2.8            0.0             3
2    2026-03-01T02:00            -2.0            0.0             3
3    2026-03-01T03:00            -2.2            0.0             3
4    2026-03-01T04:00            -2.2            0.0             3
..                ...             ...            ...           ...
739  2026-03-31T19:00            24.0            0.0             3
740  2026-03-31T20:00            23.8            0.0             3
741  2026-03-31T21:00            23.4            0.0             3
742  2026-03-31T22:00            22.8            0.0             3
743  2026-03-31T23:00            21.9            0.0             3

[744 rows x 4 columns]


26/10/14 18:20:10 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 792427 ms exceeds timeout 120000 ms
26/10/14 18:20:10 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/14 18:24:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

In [26]:
  weather_df_sp.groupBy(F.to_date("time", "yyyy-MM-dd'T'HH:mm").alias("date")).count().filter(col("count") == 24).tail(10)

[Row(date=datetime.date(2026, 3, 22), count=24),
 Row(date=datetime.date(2026, 3, 25), count=24),
 Row(date=datetime.date(2026, 3, 24), count=24),
 Row(date=datetime.date(2026, 3, 23), count=24),
 Row(date=datetime.date(2026, 3, 28), count=24),
 Row(date=datetime.date(2026, 3, 26), count=24),
 Row(date=datetime.date(2026, 3, 27), count=24),
 Row(date=datetime.date(2026, 3, 30), count=24),
 Row(date=datetime.date(2026, 3, 31), count=24),
 Row(date=datetime.date(2026, 3, 29), count=24)]